# Hospital Readmission Risk Prediction
## 03. Experiment Tracking — Scenario 2: XGBoost/LightGBM + Optuna

Per proposal 4.3: **XGBoost** and **LightGBM** with **Optuna** hyperparameter search. A model is promoted only if it exceeds the baseline on validation **PR-AUC**.

### MLflow setup (centralized tracking server)
- **Tracking**: Connect to scenario-1's UI server (port 5001); run scenario-1 first to launch it
- **UI**: http://localhost:5001 (same as scenario-1; run scenario-1 first to launch UI)
- **Metrics**: PR-AUC (primary), ROC-AUC, Recall at K

In [1]:
%pip install mlflow pandas scikit-learn xgboost lightgbm optuna "optuna-integration[mlflow]" seaborn matplotlib

  Using cached optuna-4.7.0-py3-none-any.whl.metadata (17 kB)
  Using cached optuna_integration-4.7.0-py3-none-any.whl.metadata (12 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached optuna-4.7.0-py3-none-any.whl (413 kB)
Using cached optuna_integration-4.7.0-py3-none-any.whl (99 kB)
Using cached colorlog-6.10.1-py3-none-any.whl (11 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [optuna-integration]ptuna-integration]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5001")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/IE_MLOps_Group-Project_Team3/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: http://127.0.0.1:5001


## Load Data & Prepare Features
Same logic as scenario-1 and 02: patient-level split, feature engineering.

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
import xgboost as xgb
import lightgbm as lgb
import optuna
from optuna.integration import MLflowCallback

DATA_PATH = '../data/diabetic_data.csv'
TARGET = 'target'
SEED = 42

def read_data(path, limit=None):
    df = pd.read_csv(path)
    df['target'] = df['readmitted'].isin(['30', '<30']).astype(int)
    if 'weight' in df.columns:
        df = df.drop(columns=['weight'])
    for col in ['medical_specialty', 'payer_code']:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown').replace('?', 'Unknown')
    df['age'] = df['age'].fillna('[50-60)')
    df['gender'] = df['gender'].fillna('Unknown')
    df['change'] = df['change'].fillna('No')
    df['diabetesMed'] = df['diabetesMed'].fillna('No')
    for col in ['A1Cresult', 'max_glu_serum']:
        if col not in df.columns:
            df[col] = 'not_tested'
        else:
            df[col] = df[col].fillna('None').replace('None', 'not_tested').astype(str)
    if 'race' in df.columns:
        df['race'] = df['race'].fillna('Unknown').replace('?', 'Unknown').astype(str)
    for col in ['number_emergency', 'number_inpatient', 'number_outpatient']:
        df[col] = pd.to_numeric(df.get(col, 0), errors='coerce').fillna(0).astype(int)
    df['care_intensity'] = df['number_emergency'] + df['number_inpatient'] + df['number_outpatient']
    df['medication_changed'] = (df['change'] == 'Ch').astype(int)
    for col in ['num_lab_procedures', 'num_procedures', 'num_medications', 'number_diagnoses']:
        df[col] = pd.to_numeric(df.get(col, 0), errors='coerce').fillna(0).astype(int)
    if limit:
        df = df.sample(n=min(limit, len(df)), random_state=SEED)
    return df

FEATURE_COLS = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_emergency', 'number_inpatient', 'number_outpatient', 'number_diagnoses', 'care_intensity',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'age', 'gender', 'race', 'change', 'diabetesMed', 'medication_changed', 'A1Cresult', 'max_glu_serum']

df = read_data(DATA_PATH, limit=50_000)
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
patient_target = df.groupby('patient_nbr')[TARGET].max()
train_patients, val_patients = train_test_split(
    patient_target.index.tolist(), test_size=0.2, random_state=SEED, stratify=patient_target.values
)
df_train = df[df['patient_nbr'].isin(train_patients)]
df_val = df[df['patient_nbr'].isin(val_patients)]

train_dicts = df_train[FEATURE_COLS].to_dict(orient='records')
val_dicts = df_val[FEATURE_COLS].to_dict(orient='records')
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
y_train = df_train[TARGET].values
y_val = df_val[TARGET].values

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'Train: {X_train.shape[0]:,} rows; Val: {X_val.shape[0]:,} rows')
print(f'Positive rate (train): {y_train.mean():.2%}; scale_pos_weight: {scale_pos_weight:.2f}')

Train: 40,030 rows; Val: 9,970 rows
Positive rate (train): 11.15%; scale_pos_weight: 7.97


## Optuna + XGBoost
Hyperparameter search over learning rate, max_depth, n_estimators. Optimize **PR-AUC** (primary metric).

In [4]:
def recall_at_k(y_true, y_proba, k=0.2):
    n = int(len(y_true) * k)
    top_idx = np.argsort(y_proba)[::-1][:n]
    return y_true[top_idx].sum() / max(y_true.sum(), 1)

def objective_xgb(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': scale_pos_weight,
        'random_state': SEED,
        'eval_metric': 'aucpr',
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, verbose=False)
    y_proba = model.predict_proba(X_val)[:, 1]
    pr_auc = average_precision_score(y_val, y_proba)
    return pr_auc

mlflow.set_experiment("hospital-readmission-xgboost")
mlflow_callback = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name='pr_auc',
)

study_xgb = optuna.create_study(direction='maximize', study_name='xgboost-optuna')
study_xgb.optimize(objective_xgb, n_trials=5, callbacks=[mlflow_callback], show_progress_bar=True)
print(f'Best XGBoost PR-AUC: {study_xgb.best_value:.4f}')

/var/folders/_y/x_pznj0x0m16xm40mxczxrl00000gn/T/ipykernel_68759/934256255.py:24: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
[I 2026-03-10 22:52:30,613] A new study created in memory with name: xgboost-optuna
  0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-03-10 22:52:30,894] Trial 0 finished with value: 0.2093882518151533 and parameters: {'learning_rate': 0.01665411882857955, 'max_depth': 4, 'n_estimators': 120, 'subsample': 0.6904798650872785, 'colsample_bytree': 0.7706569034466276}. Best is trial 0 with value: 0.2093882518151533.


Best trial: 0. Best value: 0.209388:  20%|██        | 1/5 [00:00<00:02,  1.82it/s]

🏃 View run 0 at: http://127.0.0.1:5001/#/experiments/3/runs/7d3d06ab8d9f4b43b0a47a59b9c83cb6
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


Best trial: 1. Best value: 0.21277:  40%|████      | 2/5 [00:00<00:01,  2.28it/s] 

...

[I 2026-03-10 22:52:31,760] Trial 2 finished with value: 0.20775760055637243 and parameters: {'learning_rate': 0.08606727889780055, 'max_depth': 6, 'n_estimators': 77, 'subsample': 0.726671210717449, 'colsample_bytree': 0.7391213036947883}. Best is trial 1 with value: 0.2127695984506506.
🏃 View run 2 at: http://127.0.0.1:5001/#/experiments/3/runs/74b16f79d4284f579001e5afabe82953
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


Best trial: 1. Best value: 0.21277:  80%|████████  | 4/5 [00:01<00:00,  2.07it/s]

[I 2026-03-10 22:52:32,401] Trial 3 finished with value: 0.20372995188152324 and parameters: {'learning_rate': 0.2686056492441208, 'max_depth': 3, 'n_estimators': 199, 'subsample': 0.9384820419005979, 'colsample_bytree': 0.9654078927116545}. Best is trial 1 with value: 0.2127695984506506.
🏃 View run 3 at: http://127.0.0.1:5001/#/experiments/3/runs/8de70c2cfa2c411284588e6c01ce91a6
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


Best trial: 1. Best value: 0.21277: 100%|██████████| 5/5 [00:02<00:00,  2.36it/s]

[I 2026-03-10 22:52:32,655] Trial 4 finished with value: 0.209795841327298 and parameters: {'learning_rate': 0.10508502880070615, 'max_depth': 3, 'n_estimators': 60, 'subsample': 0.6942057998189137, 'colsample_bytree': 0.7434469644652697}. Best is trial 1 with value: 0.2127695984506506.
🏃 View run 4 at: http://127.0.0.1:5001/#/experiments/3/runs/b9c873f122224ea0adb4d56f574e9b08
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3
Best XGBoost PR-AUC: 0.2128


## Optuna + LightGBM
Same setup for LightGBM.

In [5]:
def objective_lgb(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': scale_pos_weight,
        'random_state': SEED,
        'verbose': -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_val)[:, 1]
    pr_auc = average_precision_score(y_val, y_proba)
    return pr_auc

mlflow.set_experiment("hospital-readmission-lightgbm")
mlflow_callback_lgb = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name='pr_auc',
)

study_lgb = optuna.create_study(direction='maximize', study_name='lightgbm-optuna')
study_lgb.optimize(objective_lgb, n_trials=5, callbacks=[mlflow_callback_lgb], show_progress_bar=True)
print(f'Best LightGBM PR-AUC: {study_lgb.best_value:.4f}')

/var/folders/_y/x_pznj0x0m16xm40mxczxrl00000gn/T/ipykernel_68759/1055481184.py:19: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback_lgb = MLflowCallback(
[I 2026-03-10 22:52:32,764] A new study created in memory with name: lightgbm-optuna
  0%|          | 0/5 [00:00<?, ?it/s]/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/IE_MLOps_Group-Project_Team3/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Best trial: 0. Best value: 0.213955:  20%|██        | 1/5 [00:01<00:04,  1.01s/it]

[I 2026-03-10 22:52:33,696] Trial 0 finished with value: 0.2139547361324043 and parameters: {'learning_rate': 0.022040561751976398, 'max_depth': 6, 'n_estimators': 90, 'subsample': 0.6111948725696237, 'colsample_bytree': 0.6440799096110776}. Best is trial 0 with value: 0.2139547361324043.
🏃 View run 0 at: http://127.0.0.1:5001/#/experiments/5/runs/37d3013759264ef1bf511ffd93355a55
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5


/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/IE_MLOps_Group-Project_Team3/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Best trial: 1. Best value: 0.215927:  40%|████      | 2/5 [00:02<00:03,  1.05s/it]

[I 2026-03-10 22:52:34,786] Trial 1 finished with value: 0.21592655819794937 and parameters: {'learning_rate': 0.0327355199944686, 'max_depth': 7, 'n_estimators': 153, 'subsample': 0.933460860784798, 'colsample_bytree': 0.6741272773280317}. Best is trial 1 with value: 0.21592655819794937.
🏃 View run 1 at: http://127.0.0.1:5001/#/experiments/5/runs/38a505b0533c463692f20f91e43dd82e
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5


/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/IE_MLOps_Group-Project_Team3/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Best trial: 1. Best value: 0.215927:  60%|██████    | 3/5 [00:02<00:01,  1.37it/s]

[I 2026-03-10 22:52:35,068] Trial 2 finished with value: 0.20130556633268384 and parameters: {'learning_rate': 0.016779858834869068, 'max_depth': 3, 'n_estimators': 105, 'subsample': 0.9828436956109206, 'colsample_bytree': 0.9148756069471894}. Best is trial 1 with value: 0.21592655819794937.
🏃 View run 2 at: http://127.0.0.1:5001/#/experiments/5/runs/77708e7bf861402383afe94e1c6d82b6
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5


/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/IE_MLOps_Group-Project_Team3/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Best trial: 1. Best value: 0.215927:  80%|████████  | 4/5 [00:02<00:00,  1.70it/s]

[I 2026-03-10 22:52:35,518] Trial 3 finished with value: 0.20989921524150945 and parameters: {'learning_rate': 0.04277139418005929, 'max_depth': 5, 'n_estimators': 59, 'subsample': 0.933147595775814, 'colsample_bytree': 0.8535893507254095}. Best is trial 1 with value: 0.21592655819794937.
🏃 View run 3 at: http://127.0.0.1:5001/#/experiments/5/runs/76ead958fa724f03a1dc906afe3ed26f
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5


/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/IE_MLOps_Group-Project_Team3/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Best trial: 1. Best value: 0.215927: 100%|██████████| 5/5 [00:03<00:00,  1.47it/s]

[I 2026-03-10 22:52:36,099] Trial 4 finished with value: 0.2010081863723447 and parameters: {'learning_rate': 0.19692839226214476, 'max_depth': 4, 'n_estimators': 175, 'subsample': 0.7497189301229225, 'colsample_bytree': 0.7129551556826498}. Best is trial 1 with value: 0.21592655819794937.
🏃 View run 4 at: http://127.0.0.1:5001/#/experiments/5/runs/6e0ecc4c7f9243d6bf59dda719a44aa4
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5
Best LightGBM PR-AUC: 0.2159


## Log Best Model & Promote if Beats Baseline
Train best config, log full metrics (PR-AUC, ROC-AUC, Recall@K20), confusion matrix, and register model if it beats baseline.

In [6]:
BASELINE_PR_AUC = 0.189  # From scenario-1

best_xgb = study_xgb.best_params
best_lgb = study_lgb.best_params

# Pick champion by PR-AUC
champion_pr_auc = max(study_xgb.best_value, study_lgb.best_value)
if study_xgb.best_value >= study_lgb.best_value:
    champion_name, champion_params = 'XGBoost', best_xgb
    champion_model = xgb.XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=SEED, **best_xgb)
else:
    champion_name, champion_params = 'LightGBM', best_lgb
    champion_model = lgb.LGBMClassifier(scale_pos_weight=scale_pos_weight, random_state=SEED, verbose=-1, **best_lgb)

champion_model.fit(X_train, y_train)
y_proba_val = champion_model.predict_proba(X_val)[:, 1]
y_pred_val = champion_model.predict(X_val)

pr_auc = average_precision_score(y_val, y_proba_val)
roc_auc = roc_auc_score(y_val, y_proba_val)
rec_k20 = recall_at_k(y_val, y_proba_val, k=0.2)

mlflow.set_experiment("hospital-readmission-champion")
with mlflow.start_run(run_name=f"champion_{champion_name}") as run:
    mlflow.log_params(champion_params)
    mlflow.log_param("model_type", champion_name)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("recall_at_k20", rec_k20)
    mlflow.set_tag("beats_baseline", pr_auc > BASELINE_PR_AUC)
    
    cm = confusion_matrix(y_val, y_pred_val)
    cm_df = pd.DataFrame(cm, index=['No', 'Yes'], columns=['No', 'Yes'])
    cm_df.to_csv("confusion_matrix_champion.csv")
    mlflow.log_artifact("confusion_matrix_champion.csv")
    
    import seaborn as sns
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix ({champion_name} Champion)')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig('confusion_matrix_champion.png')
    plt.close()
    mlflow.log_artifact('confusion_matrix_champion.png')
    
    input_example = X_val[0:1]
    if hasattr(input_example, 'toarray'):
        input_example = input_example.toarray()
    if champion_name == 'XGBoost':
        mlflow.xgboost.log_model(champion_model, "model", input_example=input_example)
    else:
        mlflow.lightgbm.log_model(champion_model, "model", input_example=input_example)
    
    if pr_auc > BASELINE_PR_AUC:
        mlflow.register_model(f"runs:/{run.info.run_id}/model", "hospital-readmission-risk")
        print(f'Champion promoted: PR-AUC={pr_auc:.3f} > baseline {BASELINE_PR_AUC}')
    else:
        print(f'Champion NOT promoted: PR-AUC={pr_auc:.3f} <= baseline {BASELINE_PR_AUC}')
    print(f'Run ID: {run.info.run_id}')

/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/IE_MLOps_Group-Project_Team3/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/10 22:52:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/IE_MLOps_Group-Project_Team3/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/10 22:52:38 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/mod

Champion promoted: PR-AUC=0.216 > baseline 0.189
Run ID: 7c7d1ea29d0e4614b1058a2cf9f861ed
🏃 View run champion_LightGBM at: http://127.0.0.1:5001/#/experiments/6/runs/7c7d1ea29d0e4614b1058a2cf9f861ed
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/6
